In [1]:
%pip install hnswlib

Note: you may need to restart the kernel to use updated packages.


In [2]:
import hnswlib
import numpy as np
import time
from sklearn.neighbors import NearestNeighbors

# ===========================
# 1. Buat data 2D acak
# ===========================
np.random.seed(42)
num_elements = 1000
dim = 2
data = np.random.random((num_elements, dim)).astype(np.float32)

# Query point
query = np.array([[0.5, 0.5]], dtype=np.float32)
k = 5  # cari 5 tetangga terdekat

# ===========================
# 2. Exact NN (Brute Force)
# ===========================
nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric='euclidean')
nn.fit(data)

start = time.perf_counter()
distances, indices = nn.kneighbors(query)
end = time.perf_counter()

print("=== Exact NN ===")
print("Indices:", indices)
print("Distances:", distances)
print("Waktu:", end - start, "detik")

# ===========================
# 3. HNSW
# ===========================
# Inisialisasi index HNSW
p = hnswlib.Index(space='l2', dim=dim)  

# Ukuran maksimum elemen yang bisa ditampung
p.init_index(max_elements=num_elements, ef_construction=100, M=16)

# Tambahkan data
p.set_num_threads(4)
p.add_items(data)

# Set parameter pencarian
p.set_ef(50)   # tradeoff speed vs accuracy

start = time.perf_counter()
labels, distances = p.knn_query(query, k=k)
end = time.perf_counter()

print("\n=== HNSW ===")
print("Indices:", labels)
print("Distances:", distances)
print("Waktu:", end - start, "detik")


=== Exact NN ===
Indices: [[137 170 750 316 991]]
Distances: [[0.01144331 0.0278216  0.02824018 0.03417073 0.04514859]]
Waktu: 2.0668936999973084 detik

=== HNSW ===
Indices: [[137 170 750 316 991]]
Distances: [[0.00013095 0.00077404 0.00079751 0.00116764 0.0020384 ]]
Waktu: 0.00015699999858043157 detik


HNSW `l2` menghasilkan kuadrat jarak Euclidean, sedangkan sklearn menghasilkan jarak Euclidean.

In [3]:
import pandas as pd
from IPython.display import display

def uji_hnsw(metric='euclidean', num_elements=1000, dim=2):
    np.random.seed(42)
    data = np.random.random((num_elements, dim)).astype(np.float32)
    query = np.full((1, dim), 0.5, dtype=np.float32)
    k = 5

    nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric=metric)
    nn.fit(data)
    start = time.perf_counter()
    distances_exact, indices = nn.kneighbors(query)
    waktu_exact = time.perf_counter() - start

    p = hnswlib.Index(space='l2' if metric == 'euclidean' else 'cosine', dim=dim)
    p.init_index(max_elements=num_elements, ef_construction=100, M=16, random_seed=42)
    p.set_num_threads(4)
    p.add_items(data)
    p.set_ef(50)

    start = time.perf_counter()
    labels, distances_hnsw = p.knn_query(query, k=k)
    waktu_hnsw = time.perf_counter() - start

    idx_exact = indices[0].tolist()
    idx_hnsw = labels[0].tolist()
    return {
        'Distance Metrics': metric.capitalize(), 'Jumlah data': num_elements, 'Dimensi': dim,
        'Hasil Index terdekat Exact vs HNSW': f'{idx_exact}, {idx_hnsw}',
        'Waktu Exact vs HNSW (detik)': f'{waktu_exact:.8f}, {waktu_hnsw:.8f}'
    }

hasil_hnsw = []

In [4]:
hasil_hnsw.append(uji_hnsw('euclidean', 1000, 2))

In [5]:
hasil_hnsw.append(uji_hnsw('euclidean', 1000, 5))

In [6]:
hasil_hnsw.append(uji_hnsw('euclidean', 1000000, 2))

In [7]:
hasil_hnsw.append(uji_hnsw('euclidean', 1000000, 5))

In [8]:
hasil_hnsw.append(uji_hnsw('cosine', 1000, 2))

In [9]:
hasil_hnsw.append(uji_hnsw('cosine', 1000, 5))

In [10]:
hasil_hnsw.append(uji_hnsw('cosine', 1000000, 2))

In [11]:
hasil_hnsw.append(uji_hnsw('cosine', 1000000, 5))

In [12]:
tabel_hnsw = pd.DataFrame(hasil_hnsw)
with pd.option_context('display.max_colwidth', None):
    display(tabel_hnsw)

,Distance Metrics,Jumlah data,Dimensi,Hasil Index terdekat Exact vs HNSW,Waktu Exact vs HNSW (detik)
0,Euclidean,1000,2,"[137, 170, 750, 316, 991], [137, 170, 750, 316, 991]","0.02830160, 0.00009270"
1,Euclidean,1000,5,"[895, 451, 375, 386, 411], [895, 451, 375, 386, 411]","0.00446110, 0.00008770"
2,Euclidean,1000000,2,"[286942, 583487, 695123, 948064, 829288], [286942, 583487, 695123, 948064, 829288]","0.01759020, 0.00006070"
3,Euclidean,1000000,5,"[794791, 501340, 912100, 84056, 660781], [794791, 501340, 912100, 84056, 660781]","0.02973460, 0.00010560"
4,Cosine,1000,2,"[2, 135, 413, 210, 700], [2, 135, 413, 210, 700]","0.00751120, 0.00006930"
5,Cosine,1000,5,"[248, 386, 229, 230, 437], [248, 386, 229, 230, 437]","0.00482230, 0.00012060"
6,Cosine,1000000,2,"[469917, 383546, 49557, 317128, 87009], [87009, 643863, 663077, 935853, 127533]","0.07650820, 0.00011110"
7,Cosine,1000000,5,"[422252, 464287, 605535, 319582, 869360], [422252, 464287, 605535, 319582, 869360]","0.09785260, 0.00009860"


Waktu hanya untuk pencarian satu query, tanpa build index. Hasil HNSW bisa berbeda karena pencariannya aproksimasi. Pada cosine, semakin kecil jaraknya, semakin mirip arah vektornya. Skor cosine yang sama juga dapat menghasilkan urutan indeks berbeda.